Optimization for research and scale not involving speed

In [69]:
import numpy as np
import pandas as pd

def research(x):
    return 200000 * np.log(1 + x) / np.log(101)

def scale(x):
    return 7 * (x / 100)

rows = []
for i in range(1, 101):
    re = research(i)
    sc = scale(100 - i)
    pnl = re * sc - 50000
    rows.append({"Research": i, "Research_Val": round(re), "Scale": 100-i, "Scale_Val": round(sc, 4), "PnL": round(pnl)})

df = pd.DataFrame(rows)
top5 = df.nlargest(5, "PnL")
display(top5)


,Research,Research_Val,Scale,Scale_Val,PnL
22,23,137724,77,5.39,692330
23,24,139493,76,5.32,692101
21,22,135879,78,5.46,691900
24,25,141192,75,5.25,691259
20,21,133953,79,5.53,690759


Since pnl involve research x scale x speed, all three has to be greater than zero. Worst case scenario by investing only 1% in speed, assuming getting 0.1 multiplier as 1% is the lowest possible investment, the pnl is 23269. The guaranteed pnl is 23269. 

In [70]:
# worst case scenario: 
# 23% research, 76% scale, 1% speed - pnl = 23269
rows = []
for i in range(1, 100):
    re = research(i)
    sc = scale(99 - i)
    pnl = (re * sc * 0.1) - 50000
    rows.append({"Research": i, "Research_Val": round(re), "Scale": 99-i, "Scale_Val": round(sc, 4), "PnL": round(pnl)})

df = pd.DataFrame(rows)
top5 = df.nlargest(5, "PnL")
display(top5)

,Research,Research_Val,Scale,Scale_Val,PnL
22,23,137724,76,5.32,23269
21,22,135879,77,5.39,23239
23,24,139493,75,5.25,23234
20,21,133953,78,5.46,23138
24,25,141192,74,5.18,23138


Function that finds the best research and scale combo for each total percentage invested in research + scale, represented by i. i should range from 2 to 99. At 2 at least 1 can be invested for both research and scale. At 99 to leave 1 for speed. 

Since investing all 50000 is always optimal, the pnl takes out the 50000 at all time even if the optional speed_multiplier is not set. 

The function takes is a percentage invested in research + scale, and optional speed_multipler for speed multiplier assumption. ex. the last line is investing 99% into both research and scale, assuming a reasonable 0.1 multiplier since there's only 1% invested in speed. The result show the optimal research investment (23%), scale investment (76%), and assumed speed investment (1%) produces pnl of 23269. This pnl represents the worst case or the guaranteed pnl, we should try to beat this.

In [71]:
def best_pnl_for_rs_function(i, speed_multiplier=None):
    re_investment = np.arange(1, i)
    sc_investment = i - re_investment
    pnl = research(re_investment) * scale(sc_investment)
    if speed_multiplier is not None:
        pnl *= speed_multiplier
    pnl -= 50000
    df = pd.DataFrame({"Research": re_investment, "Scale": sc_investment, "PnL": np.round(pnl)})
    top1 = df.nlargest(1, "PnL").iloc[0]
    return int(top1.Research), int(top1.Scale), float(top1.PnL)

print(best_pnl_for_rs_function(99))
print(best_pnl_for_rs_function(99, 0.1))

(23, 76, 682689.0)
(23, 76, 23269.0)


The for loop finds the best research and scale combo at all speed investment from 1 to 98 and their respective pnl. This cell assumes a 0.9 multiplier, which is defintely not possible for low speed at 1. 

The last dataframe shows that investing 6 in research, 14 in scale, and 80 in speed has a higher pnl than the worst case, therefore investing 80 in speed is the highest speed investment possible, therefore assuming the 80% investment results in 0.9 multiplier, since anything above that results in pnl lower than worst case.

The pnl in the last few rows at which speed investment is low is not realistic since it's impossible to get 0.9 multiplier by investing 1% in speed.

In [74]:
rows_09 = []
for i in range(2, 100):
    re_pe, sc_pe, pnl = best_pnl_for_rs_function(i, 0.9)
    rows_09.append({"Research": re_pe, "Scale": sc_pe, "Speed": 100 - i, "PnL": pnl})

df_09_multiplier = pd.DataFrame(rows_09)
display(df_09_multiplier)
display(df_09_multiplier[df_09_multiplier["PnL"] > 23269])
    

,Research,Scale,Speed,PnL
0,1,1,98,-48108.0
1,1,2,97,-46215.0
2,2,2,96,-44001.0
3,2,3,95,-41002.0
4,2,4,94,-38002.0
...,...,...,...,...
93,22,73,5,574908.0
94,22,74,4,583469.0
95,23,74,3,592067.0
96,23,75,2,600744.0


,Research,Scale,Speed,PnL
18,6,14,80,24377.0
19,6,15,79,29690.0
20,7,15,78,35158.0
21,7,16,77,40835.0
22,7,17,76,46512.0
...,...,...,...,...
93,22,73,5,574908.0
94,22,74,4,583469.0
95,23,74,3,592067.0
96,23,75,2,600744.0


Assuming a 0.1 multipler for every pnl, the pnl in this table is all worse than the worst case 23269, but it's not realistic since the multiplier increases as speed investment increases.

In [ ]:
rows_01 = []
for i in range(2, 100):
    re_pe, sc_pe, pnl = best_pnl_for_rs_function(i, 0.1)
    rows_01.append({"Research": re_pe, "Scale": sc_pe, "Speed": 100 - i, "PnL": pnl})

df_01_multiplier = pd.DataFrame(rows_01)
pd.set_option('display.max_rows', None)
display(df_01_multiplier)

,Research,Scale,Speed,PnL
0,1,1,98,-49790.0
1,1,2,97,-49579.0
2,2,2,96,-49333.0
3,2,3,95,-49000.0
4,2,4,94,-48667.0
...,...,...,...,...
93,22,73,5,19434.0
94,22,74,4,20385.0
95,23,74,3,21341.0
96,23,75,2,22305.0


In [88]:
rows_05 = []
for i in range(2, 100):
    re_pe, sc_pe, pnl = best_pnl_for_rs_function(i, 0.5)
    rows_05.append({"Research": re_pe, "Scale": sc_pe, "Speed": 100 - i, "PnL": pnl})

df_05_multiplier = pd.DataFrame(rows_05)
display(df_05_multiplier)

,Research,Scale,Speed,PnL
0,1,1,98,-48949.0
1,1,2,97,-47897.0
2,2,2,96,-46667.0
3,2,3,95,-45001.0
4,2,4,94,-43335.0
...,...,...,...,...
93,22,73,5,297171.0
94,22,74,4,301927.0
95,23,74,3,306704.0
96,23,75,2,311524.0


In [92]:
# ── Rank → multiplier ─────────────────────────────────────────────────────────
 
def rank_to_multiplier(rank, n_players):
    """Rank 1 = best = 0.9, Rank n = worst = 0.1, linear in between."""
    if n_players == 1:
        return 0.9
    return 0.9 - (rank - 1) / (n_players - 1) * 0.8
 
def speed_to_multiplier(my_speed, competitor_speeds):
    """
    Insert my_speed into competitor_speeds, compute rank (ties share rank),
    return the multiplier.
    """
    all_speeds = list(competitor_speeds) + [my_speed]
    n = len(all_speeds)
    # Rank: number of players with strictly higher speed + 1
    rank = sum(s > my_speed for s in all_speeds) + 1
    return rank_to_multiplier(rank, n)

# ── Competitor distributions ───────────────────────────────────────────────────
 
def sample_competitors(dist, n_competitors, n_simulations, seed=42):
    """
    Returns array of shape (n_simulations, n_competitors).
    dist: 'uniform' | 'normal' | 'low_skew' | 'high_skew' | 'adversarial'
    Speed investments clipped to [1, 80].
    """
    rng = np.random.default_rng(seed)
    if dist == 'uniform':
        samples = rng.integers(1, 81, size=(n_simulations, n_competitors))
    elif dist == 'normal':
        samples = rng.normal(loc=40, scale=15, size=(n_simulations, n_competitors))
    elif dist == 'low_skew':   # most players invest little in speed
        samples = rng.exponential(scale=15, size=(n_simulations, n_competitors)) + 1
    elif dist == 'high_skew':  # most players invest heavily in speed
        samples = 81 - rng.exponential(scale=15, size=(n_simulations, n_competitors))
    elif dist == 'adversarial':  # everyone clusters at the optimal ~23-76 split → speed~1
        samples = rng.integers(1, 6, size=(n_simulations, n_competitors))
    else:
        raise ValueError(f"Unknown distribution: {dist}")
    return np.clip(samples, 1, 80).astype(int)

# ── Main optimizer ─────────────────────────────────────────────────────────────
 
def optimize_speed(dist, n_competitors=100, n_simulations=10_000):
    competitor_samples = sample_competitors(dist, n_competitors, n_simulations)
 
    results = []
    for my_speed in range(1, 81):
        rs_budget = 100 - my_speed
        if rs_budget < 2:
            continue
 
        # Expected multiplier across simulations
        multipliers = np.array([
            speed_to_multiplier(my_speed, competitor_samples[i])
            for i in range(n_simulations)
        ])
        avg_multiplier = multipliers.mean()
 
        r, s, pnl = best_pnl_for_rs_function(rs_budget, avg_multiplier)
        results.append({
            "Speed": my_speed,
            "Research": r,
            "Scale": s,
            "Avg_Multiplier": round(avg_multiplier, 4),
            "Expected_PnL": round(pnl),
        })
 
    df = pd.DataFrame(results)
    return df

In [93]:
# ── Run all distributions ──────────────────────────────────────────────────────
pd.set_option('display.max_rows', None)

distributions = ['uniform', 'normal', 'low_skew', 'high_skew', 'adversarial']
 
print(f"{'='*70}")
print(f"{'Distribution':<15} {'Speed':>6} {'Research':>9} {'Scale':>6} {'Multiplier':>11} {'Expected PnL':>13}")
print(f"{'='*70}")
 
all_results = {}
for dist in distributions:
    df = optimize_speed(dist)
    best = df.loc[df['Expected_PnL'].idxmax()]
    all_results[dist] = df
    print(f"{dist:<15} {int(best.Speed):>6} {int(best.Research):>9} {int(best.Scale):>6} {best.Avg_Multiplier:>11.4f} {int(best.Expected_PnL):>13,}")
 
print(f"{'='*70}")

# ── Sensitivity: top 5 for each distribution ──────────────────────────────────
print("\nTop 5 speed allocations per distribution:\n")
for dist, df in all_results.items():
    print(f"--- {dist} ---")
    print(df.nlargest(5, 'Expected_PnL').to_string(index=False))
    print()

Distribution     Speed  Research  Scale  Multiplier  Expected PnL
uniform             37        16     47      0.4703       139,964
normal              48        13     39      0.6803       162,402
low_skew            22        19     59      0.7154       333,592
high_skew           76         7     17      0.7128        26,443
adversarial          5        22     73      0.9000       574,908

Top 5 speed allocations per distribution:

--- uniform ---
 Speed  Research  Scale  Avg_Multiplier  Expected_PnL
    37        16     47          0.4703        139964
    36        16     48          0.4603        139885
    38        16     46          0.4802        139839
    35        16     49          0.4503        139654
    39        15     46          0.4901        139630

--- normal ---
 Speed  Research  Scale  Avg_Multiplier  Expected_PnL
    48        13     39          0.6803        162402
    49        13     38          0.6977        162262
    47        14     39          0.6623   